**Setup and Installation**

In [1]:
!pip install pandas numpy scikit-learn tensorflow transformers torch

**Imports and Initial Configuration**

In [2]:
import pandas as pd
import numpy as np
import re
import pickle
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Deep Learning Imports
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout

# BERT Imports
import torch
from torch.utils.data import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments


**Data Setup and Preprocessing**

In [3]:
# 1. Load & Prepare Dataset
# ----------------------------
def load_review_data(num_samples=1000):
    """Load a small sample from IMDB dataset as e-commerce-like reviews"""
    path = tf.keras.utils.get_file('aclImdb_v1.tar.gz',
                                   'http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz',
                                   extract=True)

    positive_dir = f'{path}/aclImdb/train/pos'
    negative_dir = f'{path}/aclImdb/train/neg'

    texts, labels = [], []
    for d in [(positive_dir, 1), (negative_dir, 0)]:
        for filename in tf.io.gfile.listdir(d[0]):
            with tf.io.gfile.GFile(f'{d[0]}/{filename}', 'r') as f:
                texts.append(f.read())
                labels.append(d[1])

    df = pd.DataFrame({'review_text': texts, 'sentiment': labels})
    df_pos = df[df['sentiment'] == 1].sample(n=num_samples//2, random_state=42)
    df_neg = df[df['sentiment'] == 0].sample(n=num_samples//2, random_state=42)
    df_sampled = pd.concat([df_pos, df_neg]).sample(frac=1, random_state=42).reset_index(drop=True)

    return df_sampled

In [4]:
def clean_text(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    return text

In [5]:
df = load_review_data(num_samples=1000)
df['review_cleaned'] = df['review_text'].apply(clean_text)

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [6]:
X = df['review_cleaned']
y = df['sentiment']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
print(f"Dataset Loaded. Training samples: {len(X_train)}, Testing samples: {len(X_test)}")


Dataset Loaded. Training samples: 800, Testing samples: 200


**APPROACH 1: Traditional Machine Learning (TF-IDF + SVM)**

In [9]:
# 2. SVM Model
# ----------------------------
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [10]:
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train_tfidf, y_train)

SVC(kernel='linear', probability=True)

In [11]:
y_pred_svm = svm_model.predict(X_test_tfidf)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

SVM Accuracy: 0.855


In [12]:
# Save models
pickle.dump(svm_model, open("svm_model.pkl", "wb"))
pickle.dump(tfidf_vectorizer, open("tfidf_vectorizer.pkl", "wb"))

**APPROACH 2: Deep Learning (Bi-LSTM)**

In [13]:
# 3. Bi-LSTM Model
# ----------------------------
MAX_LEN = 100
VOCAB_SIZE = 10000
EMBEDDING_DIM = 128
LSTM_UNITS = 64

In [51]:
tokenizer_dl = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer_dl.fit_on_texts(X_train)

In [55]:
X_train_seq = tokenizer_dl.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')

In [56]:
X_test_seq = tokenizer_dl.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')


In [17]:
dl_model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS, dropout=0.3, recurrent_dropout=0.3)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [64]:
dl_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
dl_model.fit(X_train_pad, y_train, validation_split=0.1, epochs=5, batch_size=32)

Epoch 1/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 26s 415ms/step - accuracy: 0.5845 - loss: 1.4658 - val_accuracy: 0.5125 - val_loss: 0.9592
Epoch 2/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 7s 285ms/step - accuracy: 0.6598 - loss: 0.6041 - val_accuracy: 0.5500 - val_loss: 0.8264
Epoch 3/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 282ms/step - accuracy: 0.8396 - loss: 0.4260 - val_accuracy: 0.6250 - val_loss: 0.8295
Epoch 4/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 361ms/step - accuracy: 0.8874 - loss: 0.3004 - val_accuracy: 0.6125 - val_loss: 0.8477
Epoch 5/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 277ms/step - accuracy: 0.9343 - loss: 0.1967 - val_accuracy: 0.5750 - val_loss: 0.8907


In [66]:
loss, acc = dl_model.evaluate(X_test_pad, y_test)
print("Bi-LSTM Accuracy:", acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6145 - loss: 0.7603
Bi-LSTM Accuracy: 0.6299999952316284


In [67]:
dl_model.save("lstm_model.h5")
pickle.dump(tokenizer_dl, open("tokenizer_dl.pkl", "wb"))


APPROACH 3: BERT Fine-Tuning (DistilBERT)

In [22]:
# 4. DistilBERT Model
# ----------------------------
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    df['review_text'].tolist(), df['sentiment'].tolist(), test_size=0.2, random_state=42, stratify=df['sentiment'].tolist()
)

In [23]:
bert_tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [24]:
train_encodings = bert_tokenizer(X_train_bert, truncation=True, padding=True, max_length=128)
test_encodings = bert_tokenizer(X_test_bert, truncation=True, padding=True, max_length=128)


In [25]:
class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item
    def __len__(self):
        return len(self.labels)

In [26]:
train_dataset = ReviewDataset(train_encodings, y_train_bert)
test_dataset = ReviewDataset(test_encodings, y_test_bert)

In [27]:
bert_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [28]:
training_args = TrainingArguments(
    output_dir='./bert_results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./bert_logs',
    logging_steps=50,
    report_to="none"
)


In [29]:
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

In [30]:
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [31]:
trainer.train()
results = trainer.evaluate()
print("DistilBERT Accuracy:", results['eval_accuracy'])


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.657700
100,0.519700


Step,Training Loss
50,0.657700
100,0.519700
150,0.310800
200,0.217900


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


DistilBERT Accuracy: 0.845


In [32]:
bert_model.save_pretrained("distilbert_model")
bert_tokenizer.save_pretrained("distilbert_tokenizer")

('distilbert_tokenizer/tokenizer_config.json',
 'distilbert_tokenizer/special_tokens_map.json',
 'distilbert_tokenizer/vocab.txt',
 'distilbert_tokenizer/added_tokens.json',
 'distilbert_tokenizer/tokenizer.json')

 **Final Prediction Test Across All Models**

In [68]:
ml_model = pickle.load(open("svm_model.pkl", "rb"))
tfidf_vectorizer = pickle.load(open("tfidf_vectorizer.pkl", "rb"))

In [69]:
dl_model = load_model("lstm_model.h5")
tokenizer = pickle.load(open("tokenizer_dl.pkl", "rb"))
MAX_LEN = 100


In [77]:
BERT_AVAILABLE = True
BERT_TRAINING_SKIPPED = False

In [71]:
if BERT_AVAILABLE and not BERT_TRAINING_SKIPPED:
    bert_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert_tokenizer")
    bert_model = DistilBertForSequenceClassification.from_pretrained("distilbert_model")

In [72]:
# 2. Text Cleaning Function
# ---------------------------
def clean_text(text):
    """Basic cleaning for ML/DL models"""
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    return text

In [73]:
# 3. Test Prediction Function
# ---------------------------
def test_prediction(text):
    print(f"\nReview: '{text}'")

    cleaned_text_lower = clean_text(text)

    # --- SVM Prediction ---
    tfidf_vector = tfidf_vectorizer.transform([cleaned_text_lower])
    pred_ml = ml_model.predict(tfidf_vector)[0]
    sentiment_ml = "Positive" if pred_ml == 1 else "Negative"
    print(f"  [ML/SVM] Sentiment: {sentiment_ml}")

    # --- Bi-LSTM Prediction ---
    sequence = tokenizer.texts_to_sequences([cleaned_text_lower])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_LEN, padding='post', truncating='post')
    pred_proba_dl = dl_model.predict(padded_sequence, verbose=0)[0][0]
    sentiment_dl = "Positive" if pred_proba_dl > 0.5 else "Negative"
    print(f"  [DL/LSTM] Sentiment: {sentiment_dl} (Confidence: {pred_proba_dl:.2f})")

    # --- DistilBERT Prediction ---
    if BERT_AVAILABLE and not BERT_TRAINING_SKIPPED:
        inputs = bert_tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
        bert_model.eval()
        with torch.no_grad():
            outputs = bert_model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1).squeeze()
        prediction_index = torch.argmax(probabilities).item()
        confidence = probabilities[prediction_index].item()
        sentiment_bert = "Positive" if prediction_index == 1 else "Negative"
        print(f"  [BERT/DistilBERT] Sentiment: {sentiment_bert} (Confidence: {confidence:.2f})")
    elif BERT_AVAILABLE and BERT_TRAINING_SKIPPED:
        print("  [BERT/DistilBERT] (Training Skipped) - Cannot perform real prediction.")

In [74]:
test_prediction("The vacuum cleaner arrived quickly and its suction power is absolutely superb! Five stars for quality and fast shipping.")


Review: 'The vacuum cleaner arrived quickly and its suction power is absolutely superb! Five stars for quality and fast shipping.'
  [ML/SVM] Sentiment: Positive
  [DL/LSTM] Sentiment: Positive (Confidence: 0.95)
  [BERT/DistilBERT] Sentiment: Positive (Confidence: 0.98)


In [75]:
test_prediction("This phone case broke after two days and the seller was completely unresponsive. A terrible product and frustrating purchase.")


Review: 'This phone case broke after two days and the seller was completely unresponsive. A terrible product and frustrating purchase.'
  [ML/SVM] Sentiment: Positive
  [DL/LSTM] Sentiment: Positive (Confidence: 0.91)
  [BERT/DistilBERT] Sentiment: Negative (Confidence: 0.97)


In [76]:
test_prediction("The customer service was excellent and I enjoyed the quick solution to my problem.")


Review: 'The customer service was excellent and I enjoyed the quick solution to my problem.'
  [ML/SVM] Sentiment: Positive
  [DL/LSTM] Sentiment: Positive (Confidence: 0.93)
  [BERT/DistilBERT] Sentiment: Positive (Confidence: 0.97)
